## Example 1: Render a template against a dict

In [ ]:
import importlib.metadata

import pydantic_handlebars as ph

# Example 1: variables, #each over a list and #if, the core of Handlebars
assert importlib.metadata.version("pydantic-handlebars") == "0.2.1"
template = "Hello {{name}}!{{#if items}} You have:{{#each items}} [{{this}}]{{/each}}{{/if}}"
out = ph.render(template, {"name": "Ada", "items": ["tea", "cake"]})
assert out == "Hello Ada! You have: [tea] [cake]", out
assert ph.render(template, {"name": "Bob", "items": []}) == "Hello Bob!"
print("Example 1 OK:", out)

## Example 2: A template compiled for a pydantic model renders model instances

In [ ]:
from pydantic import BaseModel

import pydantic_handlebars as ph


class Order(BaseModel):
    customer: str
    total: float
    lines: list[str]


# Example 2: compile once against the model, render many instances
compiled = ph.compile("{{customer}} owes {{total}} for {{#each lines}}{{this}}{{#unless @last}}, {{/unless}}{{/each}}", Order)
out = compiled.render(Order(customer="Ada", total=12.5, lines=["tea", "cake"]))
assert out == "Ada owes 12.5 for tea, cake", out
print("Example 2 OK:", out)

## Example 3: Check a template against a model's JSON schema before rendering

In [ ]:
from pydantic import BaseModel

import pydantic_handlebars as ph


class User(BaseModel):
    name: str
    age: int


# Example 3: a template that names a field the model lacks is reported, a correct one is not
schema = User.model_json_schema()
bad = ph.check_template_compatibility("{{name}} <{{email}}>", schema)
good = ph.check_template_compatibility("{{name}} is {{age}}", schema)
assert not good.issues, good
assert bad.issues and any("email" in str(issue) for issue in bad.issues), bad
print("Example 3 OK:", [str(issue) for issue in bad.issues])

## Example 4: Custom helpers and HTML escaping

In [ ]:
import pydantic_handlebars as ph

# Example 4: a registered helper; with auto_escape on, {{x}} escapes HTML and {{{x}}} does not (off by default: prompts)
env = ph.HandlebarsEnvironment(auto_escape=True)
env.register_helper("shout", lambda value, *args, **kwargs: str(value).upper())
assert env.render("{{shout word}}", {"word": "hi"}) == "HI"
escaped = env.render("{{html}}", {"html": "<b>x</b>"})
raw = env.render("{{{html}}}", {"html": "<b>x</b>"})
assert escaped != raw and "&lt;b&gt;" in escaped and raw == "<b>x</b>", (escaped, raw)
print("Example 4 OK:", escaped, raw)

## Example 5: List the variables a template depends on

In [ ]:
import pydantic_handlebars as ph

# Example 5: extract_dependencies names the top-level variables a template reads; a broken template raises
deps = ph.extract_dependencies("{{user.name}} {{#if user.admin}}{{audit.count}}{{/if}} {{#each tags}}{{this}}{{/each}}")
assert set(deps) == {"user", "audit", "tags"}, deps
try:
    ph.render("{{#if open}}never closed", {"open": True})
except ph.HandlebarsError as error:
    print("Example 5 OK:", sorted(deps), "| parse error:", type(error).__name__)
else:
    raise AssertionError("an unclosed block was accepted")